In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleMambaBlock(nn.Module):
    
    def __init__(self, d_model, d_state=16, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model)

        # 1. Input projections
        self.in_proj = nn.Linear(self.d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=4, groups=self.d_inner, padding=3)
        
        # 2. Selective SSM parameters (input-dependent)
        self.x_proj = nn.Linear(self.d_inner, self.d_state * 2 + 1, bias=False)
        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)
        
        # 3. Fixed parameters
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float().repeat(self.d_inner, 1)))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        (batch, seq_len, _) = x.shape
        
        # Initial projection and split
        x_and_res = self.in_proj(x).transpose(1, 2)
        x, res = x_and_res.split(self.d_inner, dim=1)
        
        # 1D Causal Convolution
        x = self.conv1d(x)[:, :, :seq_len]
        x = F.silu(x)
        
        # Selective Scan (Simplified for CPU)
        y = self.ssm(x.transpose(1, 2))
        
        # Gated output
        out = y * F.silu(res.transpose(1, 2))
        return self.out_proj(out)

    def ssm(self, x):
        # x shape: (B, L, D) where D = d_inner (256 in your case)
        A = -torch.exp(self.A_log) # (D, N) -> (256, 16)
        D = torch.ones(self.d_inner, device=x.device)
        
        temp = self.x_proj(x) 
        delta, B, C = torch.split(temp, [1, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(delta)) # (B, L, D) -> (1, 10, 256)
        
        batch, seq_len, d_inner = x.shape
        # State h: (Batch, D, N) -> (1, 256, 16)
        h = torch.zeros(batch, d_inner, self.d_state, device=x.device)
        ys = []
        
        for t in range(seq_len):
            # delta_t: (B, D)
            delta_t = delta[:, t, :].unsqueeze(-1) # (B, D, 1)
            
            # B_t: (B, N), C_t: (B, N)
            B_t = B[:, t, :].unsqueeze(1) # (B, 1, N)
            C_t = C[:, t, :] # (B, N)
            
            # 1. Discretize A: (B, D, N)
            # A is (D, N). delta_t is (B, D, 1). Result is (B, D, N)
            dA = torch.exp(delta_t * A.unsqueeze(0)) 
            
            # 2. Discretize B: (B, D, N)
            # x_t is (B, D). Result is (B, D, N)
            dB = delta_t * B_t * x[:, t, :].unsqueeze(-1) 
            
            # 3. Update hidden state
            h = dA * h + dB
            
            # 4. Compute output: (B, D)
            # y = h @ C_t
            y_t = torch.einsum('bdn,bn->bd', h, C_t)
            ys.append(y_t)
            
        return torch.stack(ys, dim=1) + x * D





In [4]:
# --- Execution ---
device = "cpu"
model = SimpleMambaBlock(d_model=128).to(device)
dummy_input = torch.randn(1, 10, 128).to(device) # (Batch, Seq, Dim)
output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")

Input shape: torch.Size([1, 10, 128])
Output shape: torch.Size([1, 10, 128])
